# 24h 이내 ICU transfer 분류
이 노트북은 **24h 이내 non-ICU -> ICU transfer 여부를 판별하는 딥러닝 모델**을 구현합니다.

**파이프라인**: 준비 → 데이터 로딩 → 점검 → 전처리(+CLAHE) → 분할·증강 → 모델(전이학습) → 학습 → 평가

https://wikidocs.net/book/10449

---
### 셀 0. 필요 라이브러리 설치

노트북 전체 코드를 실행하기 위해 필요한 패키지 목록입니다. 터미널 또는 노트북 셀에서 아래 명령을 실행하세요.

```bash
pip install -q numpy pandas opencv-python-headless torch torchvision pydicom gcsfs scikit-learn tqdm matplotlib
```

| 패키지 | 용도 |
|---|---|
| numpy | 수치 계산 |
| pandas | 표(DataFrame) 처리 |
| opencv-python-headless | 이미지 처리 · CLAHE |
| torch / torchvision | 딥러닝 모델 · 전이학습(DenseNet121) · 증강 |
| pydicom | DICOM 파일 읽기 (apply_voi_lut) |
| gcsfs | GCS 클라우드에서 DICOM 직접 읽기 |
| scikit-learn | 분할(GroupShuffleSplit) · 평가지표(ROC-AUC, F1 등) |
| tqdm | 학습 진행률 표시 |
| matplotlib | 시각화 · Grad-CAM 표시 |

In [1]:
#!pip install -q numpy pandas opencv-python-headless torch torchvision pydicom gcsfs scikit-learn tqdm matplotlib koreanize-matplotlib

### 셀 1. 라이브러리 임포트 · 전역 설정 

필요한 도구를 불러오고, 노트북 전체에서 쓸 상수를 정의합니다.

- **라이브러리**: `numpy`(계산) · `pandas`(표) · `cv2`(이미지·CLAHE) · `torch`(딥러닝) · `pydicom`(DICOM) · `gcsfs`(클라우드)
- **경로/라벨**: Transfer CSV→1, No transfer CSV→0, GCS 버킷 경로
- **하이퍼파라미터**: `IMG_SIZE=224`, `BATCH_SIZE=32`, `EPOCHS=5`, `SEED=43`
- **CLAHE 설정**: `USE_CLAHE`(켜기/끄기), `CLAHE_CLIP_LIMIT`(증폭 상한), `CLAHE_TILE_GRID`(타일 격자)
- **뷰 필터**: `ALLOWED_VIEWS={PA, AP}` — 정면 촬영만 사용

> `set_seed()`로 난수를 고정해 **재현성**(같은 코드→같은 결과)을 확보

In [ ]:
import os
import random
from typing import Optional

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pydicom
import gcsfs   # GCS에서 DICOM을 직접 읽기 위한 파일시스템
from concurrent.futures import ThreadPoolExecutor
# pydicom 버전에 따라 apply_voi_lut 임포트 경로가 다르므로 둘 다 대응
try:
    from pydicom.pixels import apply_voi_lut          # pydicom >= 3.0
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_voi_lut  # pydicom < 3.0

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    classification_report, confusion_matrix, roc_curve,
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── 샘플 목록 CSV (person_id, image_occurrence_id, local_path 컬럼) ──
#   양성 목록 = 1, 정상 대조 목록 = 0 으로 라벨링한다.
#   CSV의 local_path 는 '.../files/pXX/.../xxxx.dcm' 형태이며,
#   여기서 'files/...' 부분만 떼어내 GCS 경로 접두어에 붙여 실제 파일을 읽는다.
ICU_CSV = "icu_cohort_preprocessed.csv" #

# ── 데이터 루트 경로 (Google Cloud Storage) ──
#   실제 DICOM은 아래 GCS 버킷/접두어 + CSV의 'files/...' 상대경로로 구성된다.
#   예) gs://.../Datasets/MIMIC-CXR/files/p10/p10000032/s50414267/xxxx.dcm
GCS_BASE = "gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/MIMIC-CXR"

# ── 클래스 이름 (인덱스 0=no transfer, 1=transfer) ──
CLASS_NAMES = [0, 1]

# ── 하이퍼파라미터 ──
TARGET_COL  = "outcome"   # DataFrame에서 라벨이 담기는 컬럼명
BATCH_SIZE  = 64
EPOCHS      = 30
NUM_WORKERS = 0         # DataLoader 워커 수 (Windows에서는 0 권장)
SEED        = 43

# ── CLAHE(대비 제한 적응형 히스토그램 평활화) 설정 ──
#   흉부 X-ray는 뼈/연부조직 간 명암차가 커 폐 실질의 미세한 병변 대비가 낮다.
#   CLAHE는 이미지를 타일로 나눠 국소적으로 히스토그램을 펴 대비를 끌어올리되,
#   clipLimit로 과증폭(노이즈 부각)을 제한한다. (참고: HEQ<AHE<CLAHE 순 개선)
USE_CLAHE          = True     # False로 두면 CLAHE 없이 성능 비교 가능
CLAHE_CLIP_LIMIT   = 2.0      # 대비 증폭 상한 (클수록 강하게, 보통 2~3)
CLAHE_TILE_GRID    = (8, 8)   # 국소 처리 타일 격자 (8x8이 일반적)

# ── DICOM 손상 판정 임계치 ──
MIN_DICOM_SIZE = 50 * 1024  # 50KB 미만이면 다운로드 실패/손상으로 간주

# ── 촬영 방향(View Position) 필터 ──
#   PA(뒤→앞)/AP(앞→뒤)는 정면 촬영. LATERAL(측면)은 폐 판독에 불리하므로 제외.
ALLOWED_VIEWS  = {"PA", "AP"}
FILTER_BY_VIEW = True

# ── GCS 파일시스템 핸들 (익명 접근이 아니면 인증된 환경에서 자동 사용) ──
fs = gcsfs.GCSFileSystem()


def to_gcs_path(local_path: str) -> str:
    """CSV의 local_path(.../files/.../xxxx.dcm)를 GCS 전체 경로로 변환한다.

    'files/' 이후의 상대경로만 취해 GCS_BASE에 이어 붙인다.
    """
    p = str(local_path).replace("\\", "/")
    idx = p.find("/files/")
    rel = p[idx + 1:] if idx != -1 else p.lstrip("/")   # 'files/...' 부분
    return f"{GCS_BASE.rstrip('/')}/{rel}"


def set_seed(seed: int):
    """random / numpy / torch 의 난수 시드를 모두 고정해 재현성을 확보한다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # cudnn 을 결정론적으로 동작시켜 GPU 연산 결과도 재현 가능하게 함
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"디바이스: {device}")

디바이스: cuda


In [ ]:
df = pd.read_csv("icu_cohort.csv")

LAB_COLS = ['hgb','hct','wbc','plt','rbc','mchc','mch','mcv','rdw',
            'sodium','potassium','chloride','bicarbonate','anion_gap',
            'bun','creatinine','neut_pct','lymph_pct']
LAG_COLS = [c + '_lag' for c in LAB_COLS]
REQ_COLS = LAB_COLS + LAG_COLS

# sex 치환
print(df['sex'].value_counts(dropna=False))          # 실제 라벨 확인
df['sex'] = df['sex'].str.strip().str.upper().map({'MALE': 0, 'FEMALE': 1})
print(df['sex'].value_counts(dropna=False))          # NaN 생겼는지 확인

# complete-case 필터
n0 = len(df)
df = df.dropna(subset=REQ_COLS).reset_index(drop=True)
print(f"{n0} -> {len(df)}  (removed {n0 - len(df)}, {1 - len(df)/n0:.1%})")
print(df['outcome'].value_counts())

df.to_csv('icu_cohort_preprocessed.csv', index=False, encoding='utf-8-sig')

### 셀 2. 데이터 명단 만들기 (라벨링)

두 CSV를 읽어 각각 라벨 1·0을 붙이고 하나의 표로 합칩니다.
`to_gcs_path()`로 CSV의 로컬 경로를 실제 **GCS 클라우드 주소**로 변환합니다.
결과 표는 `local_path` · `person_id` · `image_occurrence_id` · `file_key` · `label` 열을 가지며, 한 줄이 X-ray 한 장입니다.

In [ ]:
def load_dataframe(
    csv_of_interest: str = ICU_CSV
) -> pd.DataFrame:
    """양성/정상 목록 CSV 두 개를 읽어 라벨링된 DataFrame을 만든다.

    각 CSV: person_id, image_occurrence_id, local_path 컬럼 보유.
    반환: local_path(GCS) / person_id / image_occurrence_id / file_key / label 컬럼.
    """
    def _load(csv_path: str) -> pd.DataFrame:
        d = pd.read_csv(csv_path)
        # CSV의 local_path -> 실제 접근할 GCS 경로로 변환
        d["local_path"] = d["local_path"].apply(to_gcs_path)
        return d

    pos = _load(csv_of_interest); pos = pos[pos["outcome"] == 1]
    neg = _load(csv_of_interest); neg = neg[neg["outcome"] == 0]
    print(f"[양성 목록] {len(pos):,} 건")
    print(f"[정상 목록] {len(neg):,} 건")

    df = pd.concat([pos, neg], ignore_index = True)

    # 파일명(확장자 포함)만 별도 컬럼으로 -> 셀 13 오답 목록에서 재사용
    df["file_key"] = df["local_path"].apply(lambda p: os.path.basename(str(p)))

    # person_id/image_occurrence_id 를 정수형으로 정리
    df["person_id"] = df["person_id"].astype(int)
    df["image_occurrence_id"] = df["image_occurrence_id"].astype(int)
    df = df.reset_index(drop=True)

    # 라벨 분포 요약
    n_pos = int((df["outcome"] == 1).sum())
    n_neg = int((df["outcome"] == 0).sum())
    print(f"[라벨링 결과]  총 {len(df):,} 건 | "
          f"양성 {n_pos:,} | 정상 {n_neg:,} | "
          f"고유 환자 {df['person_id'].nunique():,}명")
    if n_pos == 0 or n_neg == 0:
        print("  [경고] 한쪽 클래스가 0건입니다. CSV 경로를 확인하세요.")
    return df

df = load_dataframe()
print(f"\n전체 샘플: {len(df):,} | 양성 비율: {df[TARGET_COL].mean():.3f}")

### 셀 3. 데이터 검진 

학습 전 데이터 상태를 점검합니다.
- `check_data_distribution()` — 클래스 개수와 transfer 비율 확인
- `check_dicom_paths()` — 앞쪽 일부 파일의 ① 존재 ② 크기 ③ **DICOM 매직넘버('DICM')** 를 빠르게 진단

> 파일 전체가 아니라 헤더의 핵심만 확인하므로 빠릅니다.

In [ ]:
def check_data_distribution(df: pd.DataFrame) -> None:
    """클래스별 이미지 개수와 전체 양성 비율을 출력한다."""
    print("=== Dataset Distribution ===")
    for idx, cls in enumerate(CLASS_NAMES):
        count = int((df[TARGET_COL] == idx).sum())
        print(f"- {cls}: {count:,} images")
    print(f"- TOTAL: {len(df):,} images | 양성 비율 {df[TARGET_COL].mean():.3f}")


def check_dicom_paths(df: pd.DataFrame, n_check: int = 20, path_col: str = "local_path") -> None:
    """앞쪽 소량 샘플만 골라 GCS 경로/파일이 정상인지 빠르게 진단한다.

    각 파일을 4단계로 점검:
      1) GCS에 파일 존재 여부
      2) 파일 크기 (MIN_DICOM_SIZE 미만이면 손상 의심)
      3) DICOM 매직넘버 확인 (offset 128에 'DICM' 4바이트가 있어야 함)
      4) 위를 모두 통과하면 정상으로 카운트
    """
    paths = df[path_col].tolist()
    sample = paths[:n_check]
    print(f"\n총 {len(paths):,} 개 경로 | 앞 {len(sample)} 개 샘플 점검")
    print("[경로 예시]")
    for p in sample[:3]:
        print(f"  {p}")

    n_missing = n_too_small = n_bad_magic = n_ok = 0
    bad_examples = []
    for p in sample:
        # 1) GCS 파일 존재 확인
        try:
            exists = fs.exists(p)
        except Exception:
            exists = False
        if not exists:
            n_missing += 1
            if len(bad_examples) < 5:
                bad_examples.append(("없음", p))
            continue
        # 2) 파일 크기 확인
        try:
            size = fs.size(p)
        except Exception:
            n_missing += 1
            continue
        if size is None or size < MIN_DICOM_SIZE:
            n_too_small += 1
            if len(bad_examples) < 5:
                bad_examples.append((f"크기 {size}B", p))
            continue
        # 3) DICOM 매직넘버('DICM') 확인 (offset 128~132 만 읽음)
        try:
            with fs.open(p, "rb") as f:
                f.seek(128)
                magic = f.read(4)
        except Exception:
            magic = b""
        if magic != b"DICM":
            n_bad_magic += 1
            if len(bad_examples) < 5:
                bad_examples.append(("매직넘버 불일치", p))
            continue
        # 4) 모두 통과
        n_ok += 1

    print("\n[진단 결과 (샘플 기준)]")
    print(f"  정상          : {n_ok}")
    print(f"  파일 없음      : {n_missing}")
    print(f"  크기 미달      : {n_too_small}  (< {MIN_DICOM_SIZE//1024}KB)")
    print(f"  DICM 아님      : {n_bad_magic}")
    if bad_examples:
        print("\n[문제 경로 예시]")
        for reason, p in bad_examples:
            print(f"  ({reason}) {p}")
    if n_ok == 0:
        print("\n[경고] 샘플에서 유효한 DICOM이 하나도 없습니다. GCS_BASE 경로/인증을 확인하세요.")


check_data_distribution(df)
check_dicom_paths(df, n_check=20)

### 셀 4. 전처리 함수 — DICOM → 학습용 이미지

이 노트북의 **핵심 전처리** 셀입니다. `load_dicom_image()`가 DICOM 한 장을 아래 순서로 변환합니다.

| 순서 | 처리 | 설명 |
|---|---|---|
| 1 | **VOI LUT** | DICOM 헤더의 windowing을 적용해 진단용 밝기·대비로 매핑 |
| 2 | **MONOCHROME1 반전** | "값이 클수록 어둡게" 규격이면 흑백을 뒤집어 일반 표현으로 통일 |
| 3 | **Min-Max 정규화** | 픽셀을 0~255로 펴서 촬영 밝기 편차 제거 |
| 4 | **CLAHE** | 국소 대비 향상 — 폐 실질의 미세 병변을 또렷하게 |
| 5 | **Resize → 3채널** | 224×224로 통일 후 흑백을 3채널로 복제(ImageNet 백본 규격) |

> **CLAHE를 왜 여기(4단계)에 넣나?**
> 정규화로 값 범위가 0~255로 정리된 **직후**, 그리고 해상도가 아직 큰 **리사이즈 전**에 적용해야
> 미세한 디테일이 뭉개지지 않고 대비 향상 효과가 가장 큽니다.

In [ ]:
def is_valid_dicom(path: str) -> bool:
    """손상/누락 파일을 거르는 빠른 검증 (GCS 파일 크기 + DICM 매직넘버).

    셀 5의 VALIDATE_ON_GCS=True 일 때 passes_filters()를 통해 사용된다.
    """
    try:
        size = fs.size(path)
        if size is None or size < MIN_DICOM_SIZE:
            return False
        with fs.open(path, "rb") as f:
            f.seek(128)                    # DICOM 매직넘버는 offset 128에 위치
            return f.read(4) == b"DICM"
    except Exception:
        return False


def get_view_position(path: str) -> Optional[str]:
    """DICOM 헤더의 ViewPosition(0018,5101)을 대문자로 반환 (픽셀은 읽지 않음).

    ViewPosition 태그가 비어 있으면 SeriesDescription 등 다른 태그에서
    'PA'/'AP'/'LAT' 문자열을 찾아 대체 추정한다.
    """
    try:
        # GCS 파일을 열어 헤더만 읽음 (stop_before_pixels=True)
        with fs.open(path, "rb") as f:
            dcm = pydicom.dcmread(f, stop_before_pixels=True, force=True)
    except Exception:
        return None
    vp = getattr(dcm, "ViewPosition", None)
    if vp:
        return str(vp).strip().upper()
    # ViewPosition이 없으면 보조 태그에서 추정
    for attr in ("SeriesDescription", "ProtocolName", "StudyDescription"):
        val = getattr(dcm, attr, None)
        if val:
            u = str(val).upper()
            if "LAT" in u: return "LATERAL"   # 'LATERAL PA' 오판정 방지를 위해 먼저 검사
            if "PA"  in u: return "PA"
            if "AP"  in u: return "AP"
    return None


def passes_filters(path: str) -> bool:
    """유효한 DICOM이면서 (옵션 활성 시) 허용 view인지 한 번에 검사한다.

    값싼 검사(크기/매직넘버)를 먼저 하고 비싼 검사(헤더 파싱)를 나중에 한다.
    """
    if not is_valid_dicom(path):
        return False
    if not FILTER_BY_VIEW:
        return True
    return get_view_position(path) in ALLOWED_VIEWS


def apply_clahe(img_u8: np.ndarray) -> np.ndarray:
    """0~255 uint8 흑백 이미지에 CLAHE를 적용해 국소 대비를 향상한다.

    - HEQ(전역 평활화)는 밝은 영역/노이즈까지 과증폭하는 단점이 있다.
    - CLAHE는 타일 단위로 히스토그램을 펴되 clipLimit로 증폭을 제한해
      폐 실질의 미세한 병변 대비를 살리면서 노이즈 폭증을 억제한다.
    """
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,
                            tileGridSize=CLAHE_TILE_GRID)
    return clahe.apply(img_u8)


def load_dicom_image(dicom_path: str, img_size: int = IMG_SIZE) -> np.ndarray:
    """GCS의 DICOM 파일을 정규화·리사이즈된 (H, W, 3) uint8 ndarray로 변환한다.

    ImageNet 사전학습 백본은 3채널 입력을 기대하므로,
    흑백 X-ray를 동일 값으로 3채널 복제한다.
    """
    # GCS에서 파일을 열어 pydicom으로 읽음
    with fs.open(dicom_path, "rb") as f:
        dcm = pydicom.dcmread(f)

    # 1) VOI LUT(windowing) 적용 → 진단에 적합한 밝기/대비로 매핑. 실패 시 raw 픽셀 사용
    try:
        data = apply_voi_lut(dcm.pixel_array, dcm)
    except Exception:
        data = dcm.pixel_array

    # 2) MONOCHROME1은 값이 클수록 어두운 규격 → 흑백 반전으로 일반적 표현에 맞춤
    if dcm.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data

    # 3) min-max 정규화 후 0~255 uint8로 변환
    data = data.astype(np.float32)
    data -= data.min()
    if data.max() > 0:
        data /= data.max()
    data = (data * 255).astype(np.uint8)

    # 4) CLAHE로 국소 대비 향상 (리사이즈 전, 원해상도에서 적용해 디테일 보존)
    if USE_CLAHE:
        data = apply_clahe(data)

    # 5) 지정 크기로 리사이즈한 뒤 3채널로 복제
    data = cv2.resize(data, (img_size, img_size))
    return np.stack([data] * 3, axis=-1)

### 셀 5. 샘플링 — 뷰 필터 + 클래스 균형 맞추기 ⚖️

`sample_by_view()`는 데이터를 섞은 뒤 병렬로 촬영방향을 검사하고, 목표를 채우면 **조기 종료**해 불필요한 조회를 줄입니다.
결과는 **No transfer 1400 : Transfer 1400**의 균형 잡힌 데이터입니다.

> `VALIDATE_ON_GCS=True`로 두면 파일 손상 검사(`is_valid_dicom`)까지 함께 수행합니다(느려짐).

In [ ]:
# ── 클래스별 목표 개수 ──
N_POS = 1400  # Among 1428
N_NEG = 1400

VALIDATE_ON_GCS = False   # True면 파일 손상 여부까지 검사 (is_valid_dicom 활성화)

def _passes(path: str) -> bool:
    """view 필터(+옵션으로 GCS 유효성)를 통과하는지 검사."""
    if VALIDATE_ON_GCS:
        # 손상 검사 + view 검사를 함께 (passes_filters가 둘 다 처리)
        return passes_filters(path)
    if not FILTER_BY_VIEW:
        return True
    return get_view_position(path) in ALLOWED_VIEWS


def sample_by_view(df: pd.DataFrame, n_pos: int, n_neg: int) -> pd.DataFrame:
    """view(+옵션 유효성)를 먼저 통과시킨 뒤, 클래스별로 목표 개수를 채운다."""

    def pick(cls: int, target: int) -> pd.DataFrame:
        pool = (df[df[TARGET_COL] == cls]
                .sample(frac=1, random_state=SEED)
                .reset_index(drop=True))

        # 후보를 앞에서부터 병렬로 view 검사하되, 목표를 채우면 조기 종료
        kept = []
        CHUNK = 256  # 한 번에 검사할 묶음 (전체를 다 조회하지 않기 위함)
        rows = pool.to_dict("records")
        with ThreadPoolExecutor(max_workers=16) as ex:
            for start in range(0, len(rows), CHUNK):
                batch = rows[start:start + CHUNK]
                oks = list(ex.map(lambda r: _passes(r["local_path"]), batch))
                kept.extend(r for r, ok in zip(batch, oks) if ok)
                if len(kept) >= target:
                    break

        got = pd.DataFrame(kept[:target])
        if len(got) < target:
            print(f"[주의] 클래스 {cls}: view 통과 {len(got)} / 요청 {target}")
        return got

    print(f"view 우선 필터 → 개수 확정 중... "
          f"(GCS검사: {VALIDATE_ON_GCS}, view필터: {FILTER_BY_VIEW})")
    pos = pick(1, n_pos)
    neg = pick(0, n_neg)

    # concat 직후엔 클래스별로 정렬된 상태이므로 한 번 더 섞는다
    return (pd.concat([pos, neg])
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))


available_df = sample_by_view(df, N_POS, N_NEG)
print(f"최종: 양성 {(available_df[TARGET_COL]==1).sum()} | "
      f"정상 {(available_df[TARGET_COL]==0).sum()} | "
      f"총 {len(available_df)} | 양성 비율 {available_df[TARGET_COL].mean():.3f}")

### 셀 6. 샘플 육안 확인 

`visualize_samples()`로 No transfer / Transfer 샘플을 각각 4장씩 표시합니다.
전처리가 잘못돼 이미지가 새까맣거나 반전된 경우를 학습 전에 눈으로 잡아냅니다.

In [ ]:
%matplotlib inline
def visualize_samples(df: pd.DataFrame, cls_idx: int = 1, n: int = 4) -> None:
    """지정 클래스(0=NORMAL, 1=PNEUMONIA) 샘플 n장을 격자로 시각화한다."""
    cls_name = CLASS_NAMES[cls_idx]
    subset = df[df[TARGET_COL] == cls_idx]["local_path"].tolist()[:n]

    ncol = min(4, len(subset))
    nrow = (len(subset) + ncol - 1) // ncol     # 실제 장수에 맞춰 격자 계산
    plt.figure(figsize=(15, 4 * nrow))
    for i, path in enumerate(subset):
        img = load_dicom_image(path)      # (H,W,3) uint8
        plt.subplot(nrow, ncol, i + 1)
        plt.imshow(img)
        plt.title(f"{cls_name} Sample {i+1}")
        plt.axis("off")
    plt.suptitle(f"Visualization: {cls_name}", fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_samples(available_df, cls_idx=0, n=4)   # NORMAL 샘플
visualize_samples(available_df, cls_idx=1, n=4)   # PNEUMONIA 샘플

### 셀 7. 이미지 캐싱

`lru_cache`로 한 번 읽은 이미지(전처리 완료본)를 메모리에 저장해 재사용합니다.
CLAHE 연산도 이미지당 한 번만 수행되고 이후 캐시에서 바로 꺼내므로 학습 속도에 부담이 적습니다.

> 캐시에는 **증강 전 원본**이 저장되므로, 증강(`transform`)은 매번 새로 적용되어 캐싱과 충돌하지 않습니다.

In [8]:
from functools import lru_cache

@lru_cache(maxsize = 2000)
def _load_cached(path):
    """전처리 완료 이미지를 메모리에 캐싱 (에폭마다 재로딩 방지)."""
    return load_dicom_image(path)

### 셀 8. Dataset · 증강 · 환자 단위 분할 

- **Dataset** (`MIMICCXRDataset`) — 표의 한 행을 `(이미지 텐서, 라벨)`로 변환
- **증강(학습용만)** — `RandomResizedCrop` · 좌우반전 · ±12° 회전 · `ColorJitter`로 과적합 완화
- **검증용** — 증강 없이 원본 그대로(공정한 채점)
- 마지막에 **ImageNet 통계로 표준화**
- **환자(person_id) 단위 8:2 분할** (`GroupShuffleSplit`)

> 같은 환자 사진이 학습·검증에 섞이면 **데이터 누수**가 생기므로 환자별로 통째 분리합니다.
> CLAHE(고정 전처리)와 `ColorJitter`(랜덤 증강)는 목적이 다릅니다. CLAHE는 *모든* 이미지의 기본 대비를 균일하게 끌어올리고,
> ColorJitter는 *학습 시* 밝기·대비를 무작위로 흔들어 다양성을 줍니다. 둘은 상호 보완적입니다.

In [10]:
import sys
import types
sys.modules["torchaudio"] = types.ModuleType("torchaudio")

In [126]:
import numpy as np, pydicom, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor

class CXRDataset(Dataset):
    def __init__(self, paths, ids, processor):
        self.paths, self.ids, self.proc = paths, ids, processor
    
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with fs.open(self.paths[i], "rb") as f:
            ds = pydicom.dcmread(f)
        arr = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = arr.max() - arr
        lo, hi = arr.min(), arr.max()
        arr = np.zeros_like(arr) if hi <= lo else (arr - lo) / (hi - lo) * 255.0

        img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")
        px = self.proc(images=img, return_tensors="pt").pixel_values[0]  # [3,518,518]
        return px, self.ids[i]

In [127]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# ---- split: outcome stratified, person-level ----
tr, tmp = train_test_split(available_df, test_size=0.30, stratify=available_df["outcome"], random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=tmp["outcome"], random_state=SEED)

for name, d in [("train", tr), ("val", va), ("test", te)]:
    print(name, len(d), d["outcome"].mean().round(4))

# 중복 환자 없음 확인 (step1 설계상 person당 1행이지만 재확인)
assert len(set(tr.person_id) & set(va.person_id)) == 0
assert len(set(tr.person_id) & set(te.person_id)) == 0
assert len(set(va.person_id) & set(te.person_id)) == 0

train 1960 0.5
val 420 0.5
test 420 0.5


In [128]:
processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")

def make_ds(d):
    return CXRDataset(d["local_path"].tolist(),
                      d["person_id"].astype(str).tolist(),
                      processor)

ds_tr, ds_va, ds_te = make_ds(tr), make_ds(va), make_ds(te)

common = dict(batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

dl_tr = DataLoader(ds_tr, **common)
dl_va = DataLoader(ds_va, **common)
dl_te = DataLoader(ds_te, **common)

In [129]:
from tqdm import tqdm

@torch.inference_mode()
def extract(dl, model):
    embs, keys = [], []
    for px, pid in tqdm(dl):
        px = px.cuda(non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(pixel_values=px)
        embs.append(out.pooler_output.float().cpu())
        keys += list(pid)
    return torch.cat(embs).numpy(), keys

def cache_embeddings(splits, model, prefix="raddino", outdir="."):
    """splits: {"train": (df, dataloader), ...}"""
    import os
    os.makedirs(outdir, exist_ok=True)
    paths = {}
    for name, (d, dl) in tqdm(splits.items()):
        emb, keys = extract(dl, model)
        assert keys == d["person_id"].astype(str).tolist(), f"{name}: order mismatch"
        p = os.path.join(outdir, f"{prefix}_{name}.npz")
        np.savez(p, emb=emb, pid=np.array(keys), y=d["outcome"].values)
        paths[name] = p
        print(f"{name}: {emb.shape} -> {p}")
    return paths

In [ ]:
from transformers import AutoImageProcessor, AutoModel
model = AutoModel.from_pretrained("microsoft/rad-dino").eval().cuda()

splits = {"train": (tr, dl_tr), "val": (va, dl_va), "test": (te, dl_te)}
paths = cache_embeddings(splits, model, prefix="raddino")

In [131]:
tr.to_csv('icu_cohort_preprocessed_train.csv', index=False, encoding='utf-8-sig')
va.to_csv('icu_cohort_preprocessed_val.csv', index=False, encoding='utf-8-sig')
te.to_csv('icu_cohort_preprocessed_test.csv', index=False, encoding='utf-8-sig')

In [132]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

LAB_COLS = ['hgb','hct','wbc','plt','rbc','mchc','mch','mcv','rdw',
            'sodium','potassium','chloride','bicarbonate','anion_gap',
            'bun','creatinine','neut_pct','lymph_pct']
LAG_COLS = [c + '_lag' for c in LAB_COLS]

def build_tab(d, scaler=None, medians=None):
    X = d[LAG_COLS + ["age"]].copy()
    if medians is None:
        medians = X.median()
    X = X.fillna(medians).values.astype(np.float32)
    if scaler is None:
        scaler = StandardScaler().fit(X)
    X = scaler.transform(X).astype(np.float32)
    sex = d["sex"].astype(np.float32).values[:, None]
    return np.hstack([X, sex]), scaler, medians

# train 통계로만 fit
Xtr, scaler, medians = build_tab(tr)
Xva, _, _ = build_tab(va, scaler, medians)
Xte, _, _ = build_tab(te, scaler, medians)
TAB_DIM = Xtr.shape[1]

In [133]:
def make_loader(split, X, d, shuffle):
    z = np.load(f"raddino_{split}.npz")
    assert list(z["pid"]) == d["person_id"].astype(str).tolist()
    ds = TensorDataset(torch.from_numpy(z["emb"]),
                       torch.from_numpy(X),
                       torch.from_numpy(d["outcome"].values.astype(np.float32)))
    return DataLoader(ds, batch_size=32, shuffle=shuffle, num_workers=0)

dl_tr = make_loader("train", Xtr, tr, shuffle=True)
dl_va = make_loader("val",   Xva, va, shuffle=False)
dl_te = make_loader("test",  Xte, te, shuffle=False)

In [134]:
class FusionHead(nn.Module):
    def __init__(self, img_dim=768, tab_dim=TAB_DIM, proj_dim=64,
                 dropout=0.3, use_image=True):
        super().__init__()
        self.use_image = use_image
        if use_image:
            self.reduce = nn.Sequential(
                nn.Linear(img_dim, proj_dim), nn.GELU(), nn.LayerNorm(proj_dim))
        self.expand = nn.Sequential(
            nn.Linear(tab_dim, proj_dim), nn.GELU(), nn.LayerNorm(proj_dim))
        d = proj_dim * 2 if use_image else proj_dim
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, 2))
    def forward(self, img_emb, tab):
        z = self.expand(tab)
        if self.use_image:
            z = torch.cat([self.reduce(img_emb), z], dim=1)
        return self.head(z)

In [135]:
from sklearn.metrics import roc_auc_score, average_precision_score
import copy

def run_epoch(model, dl, crit, opt=None):
    
    train = opt is not None
    model.train(train)
    
    tot, correct, ys, ps = 0.0, 0, [], []
    
    with torch.set_grad_enabled(train):
        
        for emb, tab, y in dl:
            emb, tab, y = emb.cuda(), tab.cuda(), y.cuda().long()
            
            logit = model(emb, tab)
            loss = crit(logit, y)
            
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
            
            tot += loss.item() * len(y)
            correct += (logit.argmax(1) == y).sum().item()
            
            ys.append(y.cpu())
            ps.append(torch.softmax(logit, 1)[:, 1].detach().cpu())
    
    y = torch.cat(ys).numpy()
    p = torch.cat(ps).numpy()
    
    return tot/len(y), correct / max(tot, 1), roc_auc_score(y, p), average_precision_score(y, p)

In [139]:
def train_model(use_image, epochs, patience):
    
    model = FusionHead(tab_dim = TAB_DIM, use_image = use_image).cuda()
    
    pw = torch.tensor([(tr.outcome == 0).sum() / (tr.outcome == 1).sum()],
                      dtype=torch.float32).cuda()
    crit = nn.CrossEntropyLoss(label_smoothing = 0.05)
    opt = torch.optim.AdamW(model.parameters(), lr = 1e-4, weight_decay = 1e-3)
    
    best, best_state, bad = -1, None, 0
    
    for ep in range(epochs):
        print(ep)
        run_epoch(model, dl_tr, crit, opt)
        _, va_cor, va_auc, va_ap = run_epoch(model, dl_va, crit)
        
        if va_auc > best:
            best, best_state, bad = va_auc, copy.deepcopy(model.state_dict()), 0
            print(best)
            if use_image:
                torch.save(model.state_dict(), "fusion_best_model.pt")
            else:
                torch.save(model.state_dict(), "tabular_best_model.pt")
        else:
            bad += 1
            if bad >= patience:
                break
    
    model.load_state_dict(best_state)
    return best

In [ ]:
train_model(True, 100, 10)

In [ ]:
train_model(False, 100, 10)

In [ ]:
@torch.no_grad()
def evaluate_model(loader, model):
    """검증셋 전체에 대해 (실제 라벨, 예측 라벨, 확률)을 반환한다."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for emb, tab, y in loader:
        emb, tab, y = emb.cuda(), tab.cuda(), y.cuda().long()
        logit = model(emb, tab)
        probs = torch.softmax(logit, dim = 1)[:, 1]
        preds = logit.argmax(dim = 1)
        all_labels.extend(y.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


# best 체크포인트 로드 후 검증셋 평가
use_image = True
model = FusionHead(tab_dim = TAB_DIM, use_image = use_image).cuda()
model.load_state_dict(torch.load("fusion_best_model.pt", weights_only = True))
y_true, y_pred, y_prob = evaluate_model(dl_te, model)

print("[Confusion Matrix]  (행=실제, 열=예측)")
print(confusion_matrix(y_true, y_pred))
print("\n[Classification Report]")
print(classification_report(y_true, y_pred, target_names= ['0', '1'], digits=4))

# ROC-AUC는 한 클래스만 있으면 계산 불가 → 예외 처리
try:
    auroc = roc_auc_score(y_true, y_prob)
except ValueError:
    auroc = float("nan")
print(f"ROC-AUC : {auroc:.4f}")
print(f"F1      : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

In [163]:
y_true_f, y_pred_f, y_prob_f = evaluate_model(dl_te, model)

In [ ]:
# best 체크포인트 로드 후 검증셋 평가
use_image = False
model = FusionHead(tab_dim = TAB_DIM, use_image = use_image).cuda()
model.load_state_dict(torch.load("tabular_best_model.pt", weights_only = True))
y_true, y_pred, y_prob = evaluate_model(dl_te, model)

print("[Confusion Matrix]  (행=실제, 열=예측)")
print(confusion_matrix(y_true, y_pred))
print("\n[Classification Report]")
print(classification_report(y_true, y_pred, target_names= ['0', '1'], digits=4))

# ROC-AUC는 한 클래스만 있으면 계산 불가 → 예외 처리
try:
    auroc = roc_auc_score(y_true, y_prob)
except ValueError:
    auroc = float("nan")
print(f"ROC-AUC : {auroc:.4f}")
print(f"F1      : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# ROC curve 그리기
plt.figure(figsize=(5, 5))
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.plot(fpr, tpr, label=f"AUROC={auroc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")   # 무작위 분류 기준선
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC"); plt.legend()
plt.show()

In [165]:
y_true_t, y_pred_t, y_prob_t = evaluate_model(dl_te, model)

### 셀 13. 오답 분석

오답을 **확신하며 틀린 순**으로 정렬해 CSV로 저장합니다.
- **FP**: 불필요한 추가 검사 유발
- **FN**: **임상적으로 가장 위험**

> 강하게 확신했는데 틀린 사례에는 배울 점(어려운 케이스, 라벨 오류 등)이 있습니다.
> `val_df`와 `y_pred`의 인덱스 대응은 `val_loader`의 `shuffle=False`에 의존합니다.

In [ ]:
# 예측이 틀린 인덱스 추출
wrong_idx = np.where(y_pred != y_true)[0]
# 확신하며 틀린 순으로 정렬 (|prob - 0.5|가 클수록 확신이 강한 오답)
wrong_idx = sorted(wrong_idx, key=lambda i: abs(y_prob[i] - 0.5), reverse=True)

fp = int(((y_pred == 1) & (y_true == 0)).sum())   # False Positive
fn = int(((y_pred == 0) & (y_true == 1)).sum())   # False Negative
print(f"오분류: {len(wrong_idx)} / {len(y_true)} 건")
print(f"  False Positive(ICU 안 가도 되는데 감): {fp}")
print(f"  False Negative(ICU 가야 하는데 안감) : {fn}")

# 오답 상세 목록을 DataFrame으로 정리
wrong_df = pd.DataFrame({
    "person_id": te.iloc[wrong_idx]["person_id"].values,
    "image_occurrence_id":   te.iloc[wrong_idx]["image_occurrence_id"].values,
    "file_name":  te.iloc[wrong_idx]["file_key"].values,
    "true":       y_true[wrong_idx],
    "prob":       np.round(y_prob[wrong_idx], 4),
    "pred":       y_pred[wrong_idx],
    "error_type": np.where(y_pred[wrong_idx] == 1, "FP(ICU 안 가도 되는데 감)", "FN(ICU 가야 하는데 안감)"),
}).sort_values("prob", ascending=False).reset_index(drop=True)

# utf-8-sig로 저장 → Excel에서 한글 깨짐 방지
wrong_df.to_csv("val_wrong_predictions.csv", index=False, encoding="utf-8-sig")
print(f"\n오답 목록 저장: val_wrong_predictions.csv ({len(wrong_df)} 건)")
wrong_df.head(10)